## Plotting Loss curve

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

log_path = "training_logs/single_sample_log0.csv"

# Read the CSV file
df = pd.read_csv(log_path)

# Extract loss values in order (ignoring worker_id)
losses = df['loss'].values

# Create step numbers (0, 1, 2, ...)
steps = np.arange(len(losses))

# Plot the loss curve
plt.figure(figsize=(12, 6))
plt.plot(steps, losses, linewidth=1, alpha=0.7)
plt.xlabel('Step', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Curve', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print some statistics
print(f"Total steps: {len(losses)}")
print(f"Min loss: {losses.min():.4f}")
print(f"Max loss: {losses.max():.4f}")
print(f"Mean loss: {losses.mean():.4f}")
print(f"Final loss: {losses[-1]:.4f}")



## Testing script on 100 subset of MNIST

In [ ]:
import random
from torch.utils.data import Subset
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm  # Install with: pip install tqdm

# Import your project modules
from nodestore import NodeStore
from full_model import Model
from gnn_model import GNN
from quantization import Quantizer
from lookup_table import LookupTable

# --- Configuration (Must match training config) ---
COLLECTION_NAME = 'final2'  # Ensure this matches the training collection
QDRANT_URL = 'http://localhost:6333'
TOTAL_NODES = 500
INPUT_NODES = 14
OUTPUT_NODES = 10
CARDINALITY = 5
VECTOR_DIM = 56
PHASE_BINS = 256
MAG_BINS = 256
GAMMA = 1.

# Evaluation specific settings
ITERATIONS = 3  # Same as training
ACTIVATION_THRESHOLD = 0.05
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def evaluate():
    print(f"Initializing Evaluation on {DEVICE}...")

    # 1. Initialize Components (loads weights from Qdrant)
    lookup_table = LookupTable(PHASE_BINS, MAG_BINS, GAMMA, device=DEVICE)

    node_store = NodeStore(
        qdrant_url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
        lookup_table=lookup_table,
        num_total_nodes=TOTAL_NODES,
        num_input_nodes=INPUT_NODES,
        num_output_nodes=OUTPUT_NODES,
        cardinality=CARDINALITY,
        vector_dim=VECTOR_DIM,
        phase_bins=PHASE_BINS,
        mag_bins=MAG_BINS,
    )

    gnn = GNN(
        node_store=node_store,
        cardinality=CARDINALITY,
        radiation_targets=CARDINALITY,
        total_nodes=TOTAL_NODES,
        input_nodes=INPUT_NODES,
        output_nodes=OUTPUT_NODES,
        phase_bins=PHASE_BINS,
        mag_bins=MAG_BINS,
        vector_dim=VECTOR_DIM,
        iterations=ITERATIONS,
        activation_threshold=ACTIVATION_THRESHOLD,
        gamma=GAMMA,
        device=DEVICE,
        verbose=False, 
    )

    quantizer = Quantizer(
        phase_bins=PHASE_BINS,
        mag_bins=MAG_BINS,
        lookup_table=lookup_table,
        vector_dim=VECTOR_DIM,
        input_node_count=INPUT_NODES,
        device=DEVICE
    )

    model = Model(gnn, quantizer)
    model.to(DEVICE)
    model.eval() # Set to evaluation mode (though your custom modules might not use it, good practice)

    # 2. Setup Data Loader (MNIST Test Set)
    transformations = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.flatten())   
    ])
    
    test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transformations)
    
    # Select 100 random indices
    subset_indices = random.sample(range(len(test_dataset)), 100)
    test_subset = Subset(test_dataset, subset_indices)
    
    # Create DataLoader from the subset
    test_loader = DataLoader(test_subset, batch_size=1, shuffle=False)

    print(f"Model loaded. Connected to collection '{COLLECTION_NAME}'.")
    print(f"Starting evaluation on {len(test_subset)} random test images...")

    correct = 0
    total = 0

    # 3. Evaluation Loop
    with torch.no_grad(): # Disable gradient calculation for inference
        for data, target in tqdm(test_loader, desc="Evaluating"):
            data, target = data.to(DEVICE), target.to(DEVICE)
            
            # Remove batch dimension since batch_size=1 and model expects flattened input
            data = data.squeeze(0) 

            # Forward pass
            output_signal = model(data)

            # Prediction is the node with the highest activation strength
            predicted = torch.argmax(output_signal)
            
            if predicted.item() == target.item():
                correct += 1
            
            total += 1
            
            # CRITICAL: Reset the graph state (activations) between samples
            model.reset()

    # 4. Results
    accuracy = 100 * correct / total
    print(f"\nEvaluation Results:")
    print(f"Total Samples: {total}")
    print(f"Correct Predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")


In [ ]:
evaluate()